# 📈 Stock Price Prediction - Exploratory Data Analysis

1. Data loading & validation
2. Statistical summary
3. Price & volume
4. Returns analysis
5. Stationarity tests
6. Seasonality
7. Simple pattern detection

> `statsmodels` is only needed for the ACF / stationarity cells: `pip install statsmodels`

In [ ]:
import os, sys
# Find the project root (folder that contains src/) no matter where Jupyter was started
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, 'src')) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
%matplotlib inline
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from src.data_loader import StockDataLoader
from src.visualization import StockVisualizer

## 1. Data loading & validation

In [ ]:
SYMBOL, START, END = 'AAPL', '2018-01-01', pd.Timestamp.today().strftime('%Y-%m-%d')
loader = StockDataLoader(SYMBOL)
df_raw, source = loader.fetch(SYMBOL, START, END)
print(f"Data source: {source.upper()}  |  rows: {len(df_raw)}")
if source == 'synthetic':
    print("⚠️  Yahoo download failed - this is SYNTHETIC data.")

for key, value in loader.validate_data(df_raw).items():
    print(f"  {key:22s}: {value}")

## 2. Statistical summary

In [ ]:
cols = ['Open', 'High', 'Low', 'Close', 'Volume']
extra = pd.DataFrame({
    'skewness': df_raw[cols].skew(),
    'kurtosis': df_raw[cols].kurtosis(),
    'range': df_raw[cols].max() - df_raw[cols].min(),
    'cv_%': df_raw[cols].std() / df_raw[cols].mean() * 100,
    'missing': df_raw[cols].isnull().sum(),
})
display(df_raw[cols].describe())
display(extra)

## 3. Price & volume

In [ ]:
StockVisualizer(df_raw, symbol=SYMBOL).plot_price_history_interactive().show()

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=('Volume', 'Volume moving averages'))
fig.add_trace(go.Bar(x=df_raw['Date'], y=df_raw['Volume'], name='Volume'), row=1, col=1)
for period, color in [(20, 'orange'), (50, 'green')]:
    fig.add_trace(go.Scatter(x=df_raw['Date'], y=df_raw['Volume'].rolling(period).mean(),
                             name=f'MA {period}', line=dict(color=color)), row=2, col=1)
fig.update_layout(height=600, template='plotly_dark')
fig.show()

## 4. Returns analysis

In [ ]:
returns = df_raw['Close'].pct_change().dropna() * 100

fig = make_subplots(rows=2, cols=2, subplot_titles=('Distribution', 'Q-Q plot vs normal', 'Box plot', 'Autocorrelation'))
fig.add_trace(go.Histogram(x=returns, nbinsx=60, name='Returns'), row=1, col=1)

(osm, osr), (slope, intercept, _) = stats.probplot(returns, dist='norm')
fig.add_trace(go.Scatter(x=osm, y=osr, mode='markers', name='Q-Q'), row=1, col=2)
fig.add_trace(go.Scatter(x=osm, y=slope * osm + intercept, mode='lines', name='Normal line'), row=1, col=2)
fig.add_trace(go.Box(y=returns, name='Returns'), row=2, col=1)

try:
    from statsmodels.tsa.stattools import acf
    lags = 20
    fig.add_trace(go.Bar(x=list(range(lags + 1)), y=acf(returns, nlags=lags), name='ACF'), row=2, col=2)
    band = 1.96 / np.sqrt(len(returns))
    fig.add_hline(y=band, line_dash='dash', row=2, col=2)
    fig.add_hline(y=-band, line_dash='dash', row=2, col=2)
except ImportError:
    print("statsmodels not installed - skipping ACF")

fig.update_layout(height=650, template='plotly_dark', showlegend=False)
fig.show()
print(f"Daily returns: mean {returns.mean():.3f}%, std {returns.std():.3f}%, "
      f"skew {returns.skew():.2f}, excess kurtosis {returns.kurtosis():.2f}")

## 5. Stationarity tests
Prices are usually non-stationary, returns usually stationary - which is why the models predict **returns**.

In [ ]:
try:
    from statsmodels.tsa.stattools import adfuller, kpss

    def test_stationarity(series, title):
        adf_p = adfuller(series.dropna(), autolag='AIC')[1]
        kpss_p = kpss(series.dropna(), regression='c', nlags='auto')[1]
        print(f"{title:12s} ADF p={adf_p:.4f} -> {'Stationary' if adf_p < 0.05 else 'Non-stationary'} | "
              f"KPSS p={kpss_p:.4f} -> {'Stationary' if kpss_p > 0.05 else 'Non-stationary'}")

    test_stationarity(df_raw['Close'], 'Close price')
    test_stationarity(returns, 'Returns')
except ImportError:
    print("pip install statsmodels to run stationarity tests")

## 6. Seasonality (average daily return, not price level)

In [ ]:
tmp = df_raw.assign(ret=df_raw['Close'].pct_change() * 100).dropna()
fig = make_subplots(rows=2, cols=2, subplot_titles=('By month', 'By quarter', 'By weekday', 'By year'))
for (key, row, col) in [(tmp['Date'].dt.month, 1, 1), (tmp['Date'].dt.quarter, 1, 2),
                        (tmp['Date'].dt.dayofweek, 2, 1), (tmp['Date'].dt.year, 2, 2)]:
    g = tmp.groupby(key)['ret'].mean()
    fig.add_trace(go.Bar(x=g.index, y=g.values), row=row, col=col)
fig.update_layout(height=600, template='plotly_dark', showlegend=False)
fig.show()

## 7. Simple pattern detection

In [ ]:
rng = (df_raw['High'] - df_raw['Low']).replace(0, np.nan)
patterns = pd.DataFrame({
    'uptrend':     ((df_raw['Close'] > df_raw['Close'].shift(1)) & (df_raw['Close'] > df_raw['Close'].shift(2))).rolling(5).mean() > 0.6,
    'downtrend':   ((df_raw['Close'] < df_raw['Close'].shift(1)) & (df_raw['Close'] < df_raw['Close'].shift(2))).rolling(5).mean() > 0.6,
    'inside_bar':  (df_raw['High'] < df_raw['High'].shift(1)) & (df_raw['Low'] > df_raw['Low'].shift(1)),
    'outside_bar': (df_raw['High'] > df_raw['High'].shift(1)) & (df_raw['Low'] < df_raw['Low'].shift(1)),
    'doji':        ((df_raw['Close'] - df_raw['Open']).abs() / rng) < 0.1,
})
summary = pd.DataFrame({'count': patterns.sum(), 'pct': patterns.mean() * 100}).round(1)
display(summary)
print("✅ EDA complete")